# Curtain - Dissolved Oxygen (print-quality snapshots)
Derivative of `curtain_1panel.ipynb`, retargeted for **print publication**.

- Variable **`WQ_OXY_OXY`** (AED `OXY_oxy`); model-native units **mmol O$_2$ m$^{-3}$**, displayed here as
  **mg L$^{-1}$** (divide by 31.25).
- **Static snapshots** (no GIF): `render_snapshot()` looped over `SNAPSHOT_TIMES`; `render_cascade()` for the
  offset/overlapping montage.
- **600 dpi PNG + vector PDF** -> `outputs_print/`.

**Feature shown:** the **13-23 Jan 2024 bottom-water O$_2$ drawdown pulse** - basin-floor O$_2$ falls from
~6.6 to ~6.1 mg L$^{-1}$ over 9 days then rebounds, appearing as a depleted slug pooling in the deep
Central Basin trough. NB this is *oxygen depletion*, not hypoxia (values stay > 5.5 mg L$^{-1}$); the
colormap red band marks the slug, not a hypoxic threshold.

Source NC: `W:\WAMSI\1.7\SH-20251123-1.7.0\2023B-20251124150126\results\csiem_B010_20221101_20240401_WQ_WQ.nc`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import geopandas as gpd
import xarray as xr
import tfv.xarray
import cmocean  # cmo.oxy oxygen colormap
sns.set_theme(style='white', font_scale=0.9)


In [ ]:
# ============================== PARAMETERS ==============================
MODEL_NC   = Path(r'W:\WAMSI\1.7\SH-20251123-1.7.0\2023B-20251124150126\results') / 'csiem_B010_20221101_20240401_WQ_WQ.nc'
SHP_PATH   = r"../../gis/Curtain/New_Curtain_line_LL_100m.shp"
OUT_DIR    = Path('./outputs_print')

VAR          = "WQ_OXY_OXY"
MMOL_PER_MGL = 31.25                       # mmol O2 m-3 per mg/L
VAR_LABEL    = "Dissolved oxygen\n(mg/L)"
CMAP         = "RdBu"                       # red = low O2, blue = high O2
CLIM_MGL     = (5.5, 7.0)                   # mg/L colour limits (upper = 7.0)
RUN_LABEL    = "CSIEM-1.7.0 2023B"

# 13-23 Jan 2024 drawdown pulse. Single image(s) first; extend the list for a series.
SNAPSHOT_TIMES = ['2024-01-23 00:00']                       # pulse minimum (~6.08 mg/L)
# Sequential cascade across the drawdown (pre -> minimum):
CASCADE_TIMES  = ['2024-01-14 12:00', '2024-01-18 12:00',
                  '2024-01-21 00:00', '2024-01-23 00:00']

SHOW_VECTORS = True                        # full depth-current arrows (show the sloshing/cascade flow)
VEC_SCALE    = 5
VEC_WIDTH    = 0.0008
FIGSIZE      = (11, 3.4)                    # inches
DPI_PNG      = 600
SAVE_PDF     = True

# Curtain x-axis landmarks (chainage m -> label)
XTICKS  = [1000, 6000, 17060, 25000, 35000]
XLABELS = ['Sepia Dep.', 'Causeway', 'Central Basin', 'Parmelia Bank', 'Fremantle']
# ========================================================================
CLIM = tuple(v * MMOL_PER_MGL for v in CLIM_MGL)   # raw variable is in mmol/m3
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
# --- Load model + curtain polyline ---
ds = xr.open_dataset(MODEL_NC, decode_times=True, engine='netcdf4')
fv = ds.tfv

gdf = gpd.read_file(SHP_PATH)
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
polyline = np.c_[gdf.geometry.x, gdf.geometry.y]
print(f"Run: {str(ds.ResTime.values[0])[:13]} -> {str(ds.ResTime.values[-1])[:13]}  ({ds.sizes['Time']} steps)")
print(f"Curtain: {polyline.shape[0]} vertices")


In [ ]:
def _add_mgL_colorbar(fig, artist, ax):
    '''Vertical colourbar with mg/L tick labels (artist data are mmol/m3).'''
    cb = fig.colorbar(artist.patch, ax=ax, orientation="vertical", fraction=0.025, pad=0.02)
    lo, hi = CLIM_MGL
    ticks_mgL = np.arange(np.ceil(lo*2)/2, hi + 1e-6, 0.5)        # every 0.5 mg/L
    cb.set_ticks(ticks_mgL * MMOL_PER_MGL)
    cb.set_ticklabels([f"{v:.1f}" for v in ticks_mgL])
    cb.set_label(VAR_LABEL)
    return cb

def render_snapshot(time, *, ax=None, save=True, colorbar=True, title=True, compass=True):
    '''Render one dissolved-oxygen curtain at `time`. Returns (fig, ax, artist).'''
    created = ax is None
    if created:
        fig = plt.figure(figsize=FIGSIZE, constrained_layout=True)
        ax = fig.add_subplot(1, 1, 1)
    else:
        fig = ax.figure
    ax.set_facecolor('lightgrey')

    cur = fv.plot_curtain(polyline, VAR, time=time, ax=ax,
                          ec="face", cmap=CMAP, clim=CLIM, colorbar=False)
    if SHOW_VECTORS:
        fv.plot_curtain_vector(polyline, time=time, ax=ax,
                               tangential=False, scale=VEC_SCALE, color="k", width=VEC_WIDTH)
    if colorbar:
        _add_mgL_colorbar(fig, cur, ax)

    ax.set_xticks(XTICKS); ax.set_xticklabels(XLABELS, rotation=0, fontsize=12)
    ax.set_xlabel(""); ax.set_ylabel("Depth (m)", fontsize=13)
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda y, _: f"{abs(y):.0f}"))
    if compass:
        ax.text(0.02, 1.04, 'S', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')
        ax.text(0.98, 1.04, 'N', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom', ha='right')
    if title:
        ts = str(np.datetime64(time))[:16].replace('T', ' ')
        ax.set_title(f'{RUN_LABEL}  -  {ts}', fontweight='bold')
    else:
        ax.set_title('')          # clear tfv's auto-applied timestamp title

    if save and created:
        stamp = str(time).replace(' ', '_').replace(':', '').replace('-', '')[:13]
        png = OUT_DIR / f'curtain_OXY_{stamp}.png'
        fig.savefig(png, dpi=DPI_PNG, bbox_inches='tight'); print("wrote", png)
        if SAVE_PDF:
            fig.savefig(png.with_suffix('.pdf'), bbox_inches='tight'); print("wrote", png.with_suffix('.pdf'))
    return fig, ax, cur


In [ ]:
# --- Single image(s) ---
for t in SNAPSHOT_TIMES:
    render_snapshot(t)


## Cascade montage (offset/overlapping frames)
Stacked column of `CASCADE_TIMES` sharing one colour scale - reads as a time-cascade of the depleted
bottom slug forming. Trial layout; tune times/offset for the designer.

In [ ]:
def render_cascade(times=CASCADE_TIMES, save=True,
                   panel_w=0.56, panel_h=0.23, dx=0.105, dy=0.22, top0=0.955):
    """Diagonal staggered montage: each panel down-and-right of the one above, slightly
    overlapping (lower panels drawn in front). No titles; date in a top-left annotation.
    y-ticks/ylabel only on the bottom panel. Colourbar sits beside the 2nd panel."""
    n = len(times)
    fig = plt.figure(figsize=(FIGSIZE[0], FIGSIZE[1] * 0.78 * n))   # manual axes (no constrained_layout)
    cur = None
    for k, t in enumerate(times):
        left   = 0.04 + k * dx
        bottom = top0 - panel_h - k * dy
        ax = fig.add_axes([left, bottom, panel_w, panel_h], zorder=k + 1)
        ax.set_facecolor('lightgrey'); ax.patch.set_alpha(1.0)
        _, _, cur = render_snapshot(t, ax=ax, save=False, colorbar=False, title=False, compass=False)
        ts = str(np.datetime64(t))[:16].replace('T', ' ')
        ax.text(0.96, 0.10, ts, transform=ax.transAxes, fontsize=13, fontweight='bold',
                va='bottom', ha='right', zorder=20,
                bbox=dict(fc='white', ec='0.6', lw=0.5, alpha=0.9, pad=2.5))
        if k == n - 1:                      # bottom panel keeps the axes furniture
            ax.set_ylabel("Depth (m)", fontsize=13)
        else:                               # strip y + x ticks/labels on the upper panels
            ax.set_ylabel(''); ax.set_yticklabels([])
            ax.set_xlabel(''); ax.set_xticklabels([])
    # colourbar beside the 2nd panel (index 1), matched to its height
    cb_left   = 0.04 + 1 * dx + panel_w + 0.02
    cb_bottom = top0 - panel_h - 1 * dy
    cax = fig.add_axes([cb_left, cb_bottom + 0.05, 0.018, panel_h - 0.09])
    cb = fig.colorbar(cur.patch, cax=cax)
    lo, hi = CLIM_MGL; ticks = np.arange(np.ceil(lo * 2) / 2, hi + 1e-6, 0.5)
    cb.set_ticks(ticks * MMOL_PER_MGL); cb.set_ticklabels([f"{v:.1f}" for v in ticks])
    cb.set_label(VAR_LABEL)
    if save:
        png = OUT_DIR / 'curtain_OXY_cascade.png'
        fig.savefig(png, dpi=DPI_PNG, bbox_inches='tight'); print("wrote", png)
        if SAVE_PDF:
            fig.savefig(png.with_suffix('.pdf'), bbox_inches='tight'); print("wrote", png.with_suffix('.pdf'))
    return fig

# render_cascade()   # uncomment to trial the diagonal cascade
